In [1]:
import pandas as pd

In [80]:
files = ["log_backup_2025_10.json", "log_backup_2025_11.json", "log_backup_2026_1.json", "log_backup_2026_2.json", "log_backup_2026_3.json"]
errors = []
rest = []

for file in files: 
     with open(f"logs/{file}") as f:
      
        for line in f:
            if "etl failure" in line.lower():
                errors.append(line)
            elif "error" in line.lower() and "metadata" not in line.lower():
                rest.append(line)
        print(f"{file}: {len(errors)} etl failures, {len(rest)} other errors")           

log_backup_2025_10.json: 144 etl failures, 241 other errors
log_backup_2025_11.json: 278 etl failures, 457 other errors
log_backup_2026_1.json: 380 etl failures, 589 other errors
log_backup_2026_2.json: 438 etl failures, 664 other errors
log_backup_2026_3.json: 525 etl failures, 827 other errors


In [19]:
len(errors)

144

In [33]:
import json
hist={}
shorts=[]
for line in errors: 
    nn = line.lower().find("etl failure for")

    a=line[nn:]
    print(a)
    b = a.split(":")
    print(b[0])
    print(b[1])
    shorts.append(b[1].strip())
    c = b[1].strip().split(" ")
    print(c[0])
    print(c[1])
    if c[1] not in hist:
        hist[c[1]] = 0
    hist[c[1]] += 1
    print("----")

ETL failure for Licensed Real Estate Professionals in Colorado: Error Extracting Licensed Real Estate Professionals in Colorado at extract","time":"2025-10-31T10:11:07.193Z","v":0}

ETL failure for Licensed Real Estate Professionals in Colorado
 Error Extracting Licensed Real Estate Professionals in Colorado at extract","time"
Error
Extracting
----
ETL failure for Professional and Occupational Licenses in Colorado: Error Extracting Professional and Occupational Licenses in Colorado at extract","time":"2025-10-31T10:11:07.203Z","v":0}

ETL failure for Professional and Occupational Licenses in Colorado
 Error Extracting Professional and Occupational Licenses in Colorado at extract","time"
Error
Extracting
----
ETL failure for Licensed Real Estate Professionals in Colorado: Error Extracting Licensed Real Estate Professionals in Colorado at extract</li><li>Error Extracting Licensed Real Estate Professionals in Colorado</li><li>job: node /usr/local/cim/bic_etl/general/scripts/sftp_extract.j

In [34]:
shorts

['Error Extracting Licensed Real Estate Professionals in Colorado at extract","time"',
 'Error Extracting Professional and Occupational Licenses in Colorado at extract","time"',
 'Error Extracting Licensed Real Estate Professionals in Colorado at extract</li><li>Error Extracting Licensed Real Estate Professionals in Colorado</li><li>job',
 'Error Extracting Professional and Occupational Licenses in Colorado at extract","time"',
 'Error Extracting Licensed Real Estate Professionals in Colorado at extract","time"',
 'Error Loading Marijuana Tax and Fee Revenue in Colorado at load","time"',
 'Error Extracting Professional and Occupational Licenses in Colorado at extract</li><li>Error Extracting Professional and Occupational Licenses in Colorado</li><li>job',
 'Error Extracting Licensed Real Estate Professionals in Colorado at extract","time"',
 'Error Extracting Professional and Occupational Licenses in Colorado at extract","time"',
 'Error Extracting CDOT Expenses at extract","time"',
 '

In [62]:
from collections import Counter
import re

def analyze_errors(messages, top_n=25):
    # --- normalize ---
    cleaned = []
    for msg in messages:
        msg = msg.lower()
        msg = re.sub(r'[^a-z0-9\s]', ' ', msg)  # remove punctuation
        msg = re.sub(r'\s+', ' ', msg).strip()
        cleaned.append(msg)

    # --- stopwords (expand as needed) ---
    stopwords = {
        "error", "for", "in", "at", "the", "and", "to", "of",
        "with", "while", "waiting", "getconnection"
    }

    # --- tokenization ---
    token_lists = []
    for msg in cleaned:
        tokens = [w for w in msg.split() if w not in stopwords]
        token_lists.append(tokens)

    # --- ngram builders ---
    def get_ngrams(tokens, n):
        return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

    # --- counters ---
    unigrams = Counter()
    bigrams = Counter()
    trigrams = Counter()

    for tokens in token_lists:
        unigrams.update(tokens)
        bigrams.update(get_ngrams(tokens, 2))
        trigrams.update(get_ngrams(tokens, 3))

    return {
        "unigrams": unigrams.most_common(top_n),
        "bigrams": bigrams.most_common(top_n),
        "trigrams": trigrams.most_common(top_n),
        "cleand": cleaned
    }

In [ ]:
analyze_errors(rest, top_n=100)

{'unigrams': [('n', 3331),
  ('node', 849),
  ('file', 844),
  ('12', 781),
  ('line', 758),
  ('py', 757),
  ('home', 702),
  ('giddensm', 702),
  ('anaconda3', 702),
  ('lib', 702),
  ('python3', 702),
  ('site', 702),
  ('packages', 702),
  ('timeout', 648),
  ('paramiko', 594),
  ('banner', 540),
  ('process', 504),
  ('etl', 478),
  ('bic', 476),
  ('exception', 432),
  ('name', 430),
  ('child', 378),
  ('self', 378),
  ('transport', 351),
  ('colorado', 348)],
 'bigrams': [('n file', 758),
  ('py line', 757),
  ('file home', 702),
  ('home giddensm', 702),
  ('giddensm anaconda3', 702),
  ('anaconda3 lib', 702),
  ('lib python3', 702),
  ('python3 12', 702),
  ('12 site', 702),
  ('site packages', 702),
  ('packages paramiko', 594),
  ('bic etl', 476),
  ('banner n', 432),
  ('child process', 378),
  ('paramiko transport', 351),
  ('transport py', 351),
  ('timeout n', 351),
  ('node internal', 343),
  ('usr local', 338),
  ('local cim', 338),
  ('cim bic', 338),
  ('n n', 331),

In [81]:
no=0
shorts=[]
for r in rest:
    if "message" in r:
      a=r.split("message")[1].strip()
      if "stack" in a:
          b=a.split("stack")[1].strip()
          shorts.append(b)
        
    else:
       no+=1
       if "msg" not in r:
           continue
       c = r.split("msg")[1].strip()
       shorts.append(c)
       print(r)
print(f"Messages without 'message' field: {no}")

{"name":"cdor_revenue_marijuana","hostname":"OITAPP182","pid":1913374,"level":50,"msg":"Job completed with errors: FAILURE: Processing Marijuana Tax and Fee Revenue in Colorado ready for CIM.csv.sdiff.gz failed: Unexpected row length at record 1: got 17 but expected 16(jobId:8ded2c92-40c2-4528-b6db-e41cd6373379)\n","time":"2025-10-22T10:20:17.713Z","v":0}

{"name":"cdot_tops","hostname":"OITAPP182","pid":4063591,"level":50,"msg":"Exception (client): Error reading SSH protocol banner\nTraceback (most recent call last):\n  File \"/home/giddensm/anaconda3/lib/python3.12/site-packages/paramiko/transport.py\", line 2369, in _check_banner\n    buf = self.packetizer.readline(timeout)\n          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/home/giddensm/anaconda3/lib/python3.12/site-packages/paramiko/packet.py\", line 395, in readline\n    buf += self._read_timeout(timeout)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/home/giddensm/anaconda3/lib/python3.12/site-packages/paramiko/packet.p

In [82]:
len(rest)

827

In [83]:
len(shorts)

786

In [84]:
c=analyze_errors(shorts, top_n=400)


In [85]:
c.keys()

dict_keys(['unigrams', 'bigrams', 'trigrams', 'cleand'])

In [86]:
c['bigrams']

[('n file', 1570),
 ('py line', 1569),
 ('home giddensm', 1430),
 ('file home', 1429),
 ('giddensm anaconda3', 1429),
 ('anaconda3 lib', 1429),
 ('lib python3', 1429),
 ('python3 12', 1429),
 ('bic etl', 1420),
 ('12 site', 1410),
 ('site packages', 1410),
 ('child process', 1392),
 ('n childprocess', 1240),
 ('node internal', 1179),
 ('packages paramiko', 1178),
 ('usr local', 978),
 ('local cim', 978),
 ('cim bic', 978),
 ('internal child', 928),
 ('banner n', 854),
 ('etl general', 850),
 ('general scripts', 770),
 ('v 0', 727),
 ('n n', 705),
 ('paramiko transport', 697),
 ('transport py', 697),
 ('timeout n', 697),
 ('data colorado', 682),
 ('colorado gov', 682),
 ('check banner', 648),
 ('emit node', 616),
 ('node events', 616),
 ('events 524', 616),
 ('524 28', 616),
 ('28 n', 616),
 ('org apache', 567),
 ('apache http', 567),
 ('http impl', 565),
 ('n raise', 528),
 ('most recent', 527),
 ('recent call', 527),
 ('call last', 527),
 ('last n', 527),
 ('impl execchain', 495),
 ('

In [87]:
fout=open("bigrams.txt", "w")# clear file
for v in c['bigrams']:
    fout.write(f"{v}\n")
fout.close()

In [88]:
fout=open("trigrams.txt", "w")# clear file
for v in c['trigrams']:
    fout.write(f"{v}\n")
fout.close()

In [89]:
def map_bigram_to_cause(bg):
    bg = bg.lower()

    # --- SCHEMA (check FIRST) ---
    if any(x in bg for x in [
        "row length", "unexpected row", "expected", "mismatch",
        "missing column", "new column", "column count", "invalid column"
    ]):
        return "SCHEMA_MISMATCH"

    # --- FILE ISSUES ---
    if any(x in bg for x in [
        "file not", "invalid file", "get file", "download",
        "cannot read file", "bad file", "corrupt"
    ]):
        return "FILE_ISSUE"

    # --- DNS / HOST ---
    if any(x in bg for x in [
        "unknownhost", "name not", "service not", "not known"
    ]):
        return "DNS_HOST_ERROR"

    # --- CONNECTION / TIMEOUT ---
    if any(x in bg for x in [
        "timed out", "timeout", "handshake", "ssh",
        "banner", "socket timeout", "read timeout", "connect timed"
    ]):
        return "CONNECTION_TIMEOUT"

    # --- API ---
    if any(x in bg for x in [
        "retrying request", "failed retrying", "processing request",
        "https", "internal error", "server error", "request failed"
    ]):
        return "API_FAILURE"

    # --- AUTH ---
    if any(x in bg for x in [
        "permission denied", "access rights", "auth"
    ]):
        return "AUTH_ERROR"

    # --- GIT / INFRA ---
    if any(x in bg for x in [
        "git pull", "repository", "origin main", "remote repository"
    ]):
        return "INFRA_ERROR"

    return "OTHER"

In [91]:
from collections import Counter

cause_counts = Counter()

for bg, count in c['bigrams']:   # your list of tuples
    cause = map_bigram_to_cause(bg)
    cause_counts[cause] += count

print(cause_counts)

Counter({'OTHER': 85632, 'CONNECTION_TIMEOUT': 8256, 'API_FAILURE': 2093, 'DNS_HOST_ERROR': 749, 'INFRA_ERROR': 452, 'FILE_ISSUE': 149, 'AUTH_ERROR': 70})


In [98]:
files = ["log_backup_2025_10.json", "log_backup_2025_11.json", "log_backup_2026_1.json", "log_backup_2026_2.json", "log_backup_2026_3.json"]
errors = []
rest = []
hist={}
keys = []
for file in files: 
     with open(f"logs/{file}") as f:
        errors = []
        rest = []
        for line in f:
            if "etl failure" in line.lower():
                errors.append(line)
            elif "error" in line.lower() and "metadata" not in line.lower():
                rest.append(line)
        print(f"{file}: {len(errors)} etl failures, {len(rest)} other errors")    

        no=0
        shorts=[]
        for r in rest:
            if "message" in r:
                a=r.split("message")[1].strip()
                if "stack" in a:
                    b=a.split("stack")[1].strip()
                    shorts.append(b)
                
            else:
                no+=1
                if "msg" not in r:
                    continue
                c = r.split("msg")[1].strip()
                shorts.append(c)
        c=analyze_errors(shorts, top_n=400)
        cause_counts = Counter()
        dt = file[11:18]
        for bg, count in c['bigrams']:   # your list of tuples
            cause = map_bigram_to_cause(bg)
            cause_counts[cause] += count

        print(cause_counts)
        for k in cause_counts.keys():
            if k not in keys:
                keys.append(k)
        hist[dt] = cause_counts
           
     

log_backup_2025_10.json: 144 etl failures, 241 other errors
Counter({'OTHER': 30877, 'CONNECTION_TIMEOUT': 4214, 'INFRA_ERROR': 116, 'FILE_ISSUE': 54, 'API_FAILURE': 28, 'AUTH_ERROR': 14})
log_backup_2025_11.json: 134 etl failures, 216 other errors
Counter({'OTHER': 35113, 'CONNECTION_TIMEOUT': 3020, 'API_FAILURE': 1985, 'DNS_HOST_ERROR': 749, 'INFRA_ERROR': 96, 'FILE_ISSUE': 32, 'AUTH_ERROR': 16})
log_backup_2026_1.json: 102 etl failures, 132 other errors
Counter({'OTHER': 7397, 'INFRA_ERROR': 96, 'CONNECTION_TIMEOUT': 16, 'AUTH_ERROR': 16, 'API_FAILURE': 16, 'FILE_ISSUE': 2})
log_backup_2026_2.json: 58 etl failures, 75 other errors
Counter({'OTHER': 5371, 'INFRA_ERROR': 96, 'CONNECTION_TIMEOUT': 16, 'AUTH_ERROR': 16, 'API_FAILURE': 16})
log_backup_2026_3.json: 87 etl failures, 163 other errors
Counter({'OTHER': 13145, 'CONNECTION_TIMEOUT': 1064, 'FILE_ISSUE': 204, 'API_FAILURE': 48, 'INFRA_ERROR': 48, 'AUTH_ERROR': 8})


In [102]:
for k in keys:
    if k != "OTHER":
       print(f"{k:<20s}", end="")
print()
for dt,cts in hist.items():
    print(f"{dt}:",end="")
    for k in keys:
        if k != "OTHER":
            print(f"   {cts.get(k, 0):<5d}", end="")
    print()

CONNECTION_TIMEOUT  FILE_ISSUE          INFRA_ERROR         API_FAILURE         AUTH_ERROR          DNS_HOST_ERROR      
2025_10:   4214    54      116     28      14      0    
2025_11:   3020    32      96      1985    16      749  
2026_1.:   16      2       96      16      16      0    
2026_2.:   16      0       96      16      16      0    
2026_3.:   1064    204     48      48      8       0    
